In [ ]:
"""
bweight_time.ipynb

Analyze bweight trajectories over time.

Author: Stellina X. Ao
Created: 2026-07-25
Last Modified: 2026-07-25
Python Version: 3.11.14
"""

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
)

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mf",
)

In [ ]:
plt.figure()
plt.imshow(encoder.tvs[:90, :15], interpolation="none", aspect="auto", cmap="coolwarm")
plt.colorbar()
plt.xlabel("response regressors")
plt.ylabel("samples (trials & bins)")
plt.tight_layout()
plt.show()

In [ ]:
encoder.verify()
encoder_mb.verify()
encoder_mf.verify()

## bweight strategy scatter

In [ ]:
from matplotlib.collections import LineCollection


def plot_trajectory(
    x,
    y,
    x_ci=None,
    y_ci=None,
    mn=None,
    mx=None,
    xlabel="",
    ylabel="",
    title="",
    cmap="plasma",
    s=2,
    start_marker="*",
    end_marker="s",
    mid_marker=".",
    start_size_mult=10,
    end_size_mult=2,
    ci_color="#333333",
    ci_alpha=0.5,
    ci_linewidth=0.5,
    ax=None,
):
    if ax is None:
        _, ax = plt.subplots(figsize=(2, 2), tight_layout=True)

    t = np.arange(len(x))
    vmin, vmax = t.min(), t.max()

    # error bars first, so everything else draws on top
    if (x_ci is not None) or (y_ci is not None):
        ax.errorbar(
            x,
            y,
            xerr=x_ci,
            yerr=y_ci,
            fmt="none",
            ecolor=ci_color,
            alpha=ci_alpha,
            elinewidth=ci_linewidth,
            capsize=0,
            zorder=1,
        )

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    lc = LineCollection(segments, cmap=cmap, array=t[:-1], linewidth=1, zorder=2)
    lc.set_clim(vmin, vmax)
    ax.add_collection(lc)

    # middle points
    if len(x) > 2:
        ax.scatter(
            x[1:-1],
            y[1:-1],
            c=t[1:-1],
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=s,
            marker=mid_marker,
            zorder=3,
        )

    # start and end, drawn on top with distinct markers
    ax.scatter(
        x[0],
        y[0],
        c="#ffffff",
        s=s * start_size_mult,
        marker=start_marker,
        zorder=4,
        edgecolors="k",
        linewidths=0.5,
    )
    ax.scatter(
        x[-1],
        y[-1],
        c="#ffffff",
        s=s * end_size_mult,
        marker=end_marker,
        zorder=4,
        edgecolors="k",
        linewidths=0.5,
    )

    # axes
    ax.axvline(x=0, color="k", linewidth=0.5, zorder=-2)
    ax.axhline(y=0, color="k", linewidth=0.5, zorder=-2)

    # lim
    if (mn is not None) and (mx is not None):
        ax.set_xlim([mn, mx])
        ax.set_ylim([mn, mx])

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    return ax

In [ ]:
from sg.models import Bootstrapper

bs_mb = Bootstrapper(
    subj_id,
    sess_id,
    make_tre(StrategyEncoder, tr_type="dme"),
    n=20,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mb",
)
bs_mf = Bootstrapper(
    subj_id,
    sess_id,
    make_tre(StrategyEncoder, tr_type="dme"),
    n=20,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mf",
)

bs_mb.get_bweight_stats()
bs_mf.get_bweight_stats()

In [ ]:
# mb v mf (mf won't be resampling anything)
plt.figure()
plt.imshow(bs_mb.idxs)
plt.show()

In [ ]:
regressor = "response_prev"
reg = "DMS"
unit_idx = encoder.reg_idxs[reg][0]

(idx_b0, idx_bm1) = (
    encoder.dm_idxs[f"{regressor}_0"],
    encoder.dm_idxs[f"{regressor}_{encoder.num_bins - 1}"],
)

ax = plot_trajectory(
    x=bs_mb.bweight_stats["mu"][unit_idx, idx_b0 : idx_bm1 + 1].T,
    y=bs_mf.bweight_stats["mu"][unit_idx, idx_b0 : idx_bm1 + 1].T,
    x_ci=bs_mb.bweight_stats["ci_h"][unit_idx, idx_b0 : idx_bm1 + 1].T,
    y_ci=bs_mf.bweight_stats["ci_h"][unit_idx, idx_b0 : idx_bm1 + 1].T,
    xlabel="mb",
    ylabel="mf",
    title=rf"$\beta$ {regressor}",
)
# ax.set_xlim([-0.05, 0.05])
# ax.set_ylim([-0.05, 0.05])

In [ ]:
plt.figure()
plt.imshow(
    bs_mb.bweight_stats["ci_h"][:, idx_b0 : idx_bm1 + 1],
    interpolation="none",
    aspect="auto",
)
plt.colorbar()
plt.show()

In [ ]:
from core.viz import plot_trajectory

regressor = "response_prev"
reg = "DMS"

idx_bounds = (
    encoder.dm_idxs[f"{regressor}_0"],
    encoder.dm_idxs[f"{regressor}_{encoder.num_bins - 1}"],
)

ax = plot_trajectory(
    x=(
        encoder_mb.encoder_weights[
            encoder.reg_idxs[reg], idx_bounds[0] : idx_bounds[1] + 1
        ].T
    ).mean(axis=1),
    y=(
        encoder_mf.encoder_weights[
            encoder.reg_idxs[reg], idx_bounds[0] : idx_bounds[1] + 1
        ].T
    ).mean(axis=1),
    xlabel="mb",
    ylabel="mf",
    title=rf"$\beta$ {regressor}",
)

In [ ]:
encoder.tbin_centers

In [ ]:
encoder.norm

In [ ]:
from squiggs.renderers import StrategyWeightPETHRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond, tv_pos_neg

mode = "response"
regressor = mode
reg = "DMS"
# rewd 33, dms
# resp 30, 36

r = StrategyWeightPETHRenderer(
    bootstrapper_mb=bs_mb,
    bootstrapper_mf=bs_mf,
    encoder_mb=encoder_mb,
    encoder_mf=encoder_mf,
    reg=reg,
    regressor=regressor,
    values=(tv_pos_neg[regressor]["pos"], tv_pos_neg[regressor]["neg"]),
    peths_mb=get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
    peths_mf=get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    pres=encoder.tpre,
    posts=encoder.tpost,
    binwidth_s=encoder.stepsize_s,
    tbin_centers=encoder.tbin_centers,
)


nv = NeuronViewer(num_units=encoder.psths[reg].shape[0], render_func=r)

In [ ]:
diff = bs_mb.bweight_stats["mu"] - bs_mf.bweight_stats["mu"]
plt.figure()
plt.imshow(
    diff[:, 5:], interpolation="none", aspect="auto", cmap="coolwarm", vmin=-1, vmax=1
)
plt.colorbar()
plt.show()

In [ ]:
diff.shape

In [ ]:
from core.viz import plot_kdes

for regr in encoder.tv_keys:
    diff_reg = {
        reg: diff[
            encoder.reg_idxs[reg],
            encoder.dm_idxs[f"{regr}_0"] : encoder.dm_idxs[
                f"{regr}_{encoder.num_bins - 1}"
            ]
            + 1,
        ].ravel()
        for reg in encoder.regions
    }

    plot_kdes(diff_reg)

#### aggregate

In [ ]:
from core.data import subject_ids, session_ids

encoders = {}

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    encoders[subj_id] = {}

    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(">", sess_id)
        try:
            encoder_mb = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.25,
                strategy_filter="mb",
            )
            encoder_mb.fit_encoder()

            encoder_mf = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.25,
                strategy_filter="mf",
            )
            encoder_mf.fit_encoder()

            encoder = make_tre(Encoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.25,
            )
            encoder.fit_encoder()

            encoders[subj_id][sess_id] = {
                "full": encoder,
                "mb": encoder_mb,
                "mf": encoder_mf,
            }

        except ValueError:
            print("blegh")
            continue

In [ ]:
from utils.viz_utils import center_title

subj_id = "MR82"
regressor = "rewarded_corr"


def plot_trajectory_sess(subj_id="MR82", regressor="rewarded_corr"):
    fig, axes = plt.subplots(
        ncols=len(encoders[subj_id]),
        nrows=2,
        figsize=(12, 4),
        sharey=True,
        sharex=True,
        tight_layout=True,
    )

    for i, encoders_sess in enumerate(encoders[subj_id].values()):
        encoder = encoders_sess["full"]
        encoder_mb = encoders_sess["mb"]
        encoder_mf = encoders_sess["mf"]

        for j, reg in enumerate(["DMS", "DLS"]):
            try:
                ax = plot_trajectory(
                    x=encoder_mb.encoder_weights[
                        :, encoder.reg_idxs[reg], encoder_mb.dm_idxs[regressor]
                    ].mean(axis=1),
                    y=encoder_mf.encoder_weights[
                        :, encoder.reg_idxs[reg], encoder_mf.dm_idxs[regressor]
                    ].mean(axis=1),
                    ax=axes[j][i],
                )
            except KeyError:
                continue

            if i == 0:
                ax.set_ylabel("mf")
                ax.set_title(reg, loc="left")
            if j == 1:
                ax.set_xlabel("mb")

    center_title(fig, axes[0], rf"$\beta$ {regressor}", fontsize=12)

    norm_str = "norm" if encoder.norm else "nonorm"
    fpath = (
        FIGURES_DIR / "time_resolved" / "bweight" / "trajectory" / norm_str / subj_id
    )
    save_fig(fig, fpath, fname=f"{regressor}_avg.png")
    return axes

In [ ]:
from core.data import tv_vals

for subj_id in ["MR82", "MR83"]:
    for regr in encoder.tv_keys:
        if regr != "response_prev":
            regressor = f"{regr}_{tv_vals[regr][0]}"
            plot_trajectory_sess(subj_id=subj_id, regressor=regressor)
        else:
            for val in tv_vals[regr]:
                regressor = f"{regr}_{val}"
                plot_trajectory_sess(subj_id=subj_id, regressor=regressor)

#### cluster

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

regressor = "response_left"

weights = np.concatenate(
    (
        encoder_mb.encoder_weights[:, :, encoder.dm_idxs[regressor]],
        encoder_mf.encoder_weights[:, :, encoder.dm_idxs[regressor]],
    )
).T

n_clusters = range(2, encoder.num_units // 2)
seeds = range(10)
silhouettes = np.zeros((len(n_clusters), len(seeds)))

for i, n in enumerate(n_clusters):
    for j, seed in enumerate(seeds):
        km = KMeans(n_clusters=n, random_state=seed)
        cluster_labels = km.fit_predict(weights)
        silhouettes[i][j] = silhouette_score(weights, cluster_labels)

n = n_clusters[np.argmax(silhouettes.mean(axis=1))]
print("cluster w/ best avg. silhouette score: ", n)

In [ ]:
plt.figure()
plt.plot(n_clusters, silhouettes.mean(axis=1))
plt.show()

In [ ]:
km = KMeans(n_clusters=5, random_state=0)
labels = km.fit_predict(weights)

labels.shape

In [ ]:
# plot the matrices instead

In [ ]:
mode = "rewarded"
reg = "DLS"
cluster_idx = 1

unit_idxs = np.intersect1d(np.where(labels == cluster_idx)[0], encoder.reg_idxs[reg])
print(len(unit_idxs), "/", encoder.psths[reg].shape[0])

if len(unit_idxs) != 0:
    r = StrategyWeightPETHRenderer(
        weights_mb=encoder_mb.encoder_weights[:, unit_idxs, :],
        weights_mf=encoder_mf.encoder_weights[:, unit_idxs, :],
        regressor="rewarded_corr",
        dm_idxs=encoder.dm_idxs,
        peths_mb=get_psths_cond(
            encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode
        ),
        peths_mf=get_psths_cond(
            encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode
        ),
        pres=encoder.tpre,
        posts=encoder.tpost,
        binwidth_s=encoder.stepsize_s,
        tbin_centers=encoder.tbin_centers,
    )

    nv = NeuronViewer(num_units=len(unit_idxs), render_func=r)

### scatter

In [ ]:
import numpy as np
from core.viz import plot_scatter, plot_2d_row
from utils.paths import FIGURES_DIR
from utils.viz_utils import save_fig

for regr in encoder.tv_keys:
    norm_str = "norm" if encoder.norm else "nonorm"
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / norm_str

    vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

    for val in vals:
        regressor = f"{regr}_{val}"
        print(regressor)
        weights = {
            reg: [
                np.concatenate(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for encoder_ in [encoder_mb, encoder_mf]
            ]
            for reg in encoder.regions
        }

        idxs = {
            reg: np.tile(range(encoder.num_bins), encoder.psths[reg].shape[0])
            for reg in encoder.regions
        }

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            color=idxs,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(
            fig,
            fpath / "scatter" / regressor / subj_id,
            f"{regressor}-{subj_id}_{sess_id}.png",
        )

## avg bweight traces

In [ ]:
import numpy as np


def get_bw_stats(encoder):
    bw_mean = {}
    bw_std = {}
    for reg in encoder.regions:
        bw_mean[reg] = np.abs(
            encoder.encoder_weights[:, encoder.reg_idxs[reg], :]
        ).mean(axis=1)
        bw_std[reg] = np.abs(encoder.encoder_weights[:, encoder.reg_idxs[reg], :]).std(
            axis=1
        )
    return bw_mean, bw_std

In [ ]:
from core.data import tv_vals
from utils.colors import colors_strategy


def plot_bw_traces(encoders, bw_stats):
    encoder = encoders["full"]
    fig, axes = plt.subplots(
        nrows=len(encoder.tv_keys) + 2,
        ncols=len(encoder.regions),
        figsize=(6, 12),
        sharex=True,
        sharey=True,
        tight_layout=True,
    )

    for j, reg in enumerate(encoder.regions):
        i = 0
        for regr in encoder.tv_keys:
            if regr != "response_prev":
                for k, (bw_mean, bw_std) in bw_stats.items():
                    ax = axes[i][j]
                    try:
                        m = bw_mean[reg][
                            :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][0]}"]
                        ]
                        s = bw_std[reg][
                            :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][0]}"]
                        ]
                    except KeyError:
                        try:
                            m = bw_mean[reg][
                                :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][1]}"]
                            ]
                            s = bw_std[reg][
                                :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][1]}"]
                            ]
                        except KeyError:
                            continue

                    ax.plot(
                        encoders[k].tbin_centers,
                        m,
                        color=colors_strategy[k],
                        label=f"{regr}, {k}",
                    )
                    ax.fill_between(
                        encoders[k].tbin_centers,
                        m - s,
                        m + s,
                        color=colors_strategy[k],
                        alpha=0.25,
                    )
                    ax.axhline(y=0, linewidth=0.5, color="k")
                    ax.axvline(x=0, linewidth=0.5, color="k")

                    ax.set_ylim([-0.2, 0.6])
                    ax.legend(loc="upper right")
                    if i == len(encoder.tv_keys) + 2 - 1:
                        ax.set_xlabel("Trial Time (s)")
                    if j == 0:
                        ax.set_ylabel(r"$\beta$")
                    if i == 0:
                        ax.set_title(reg)
                i += 1
            else:
                for val in tv_vals[regr]:
                    for k, (bw_mean, bw_std) in bw_stats.items():
                        ax = axes[i][j]
                        try:
                            m = bw_mean[reg][:, encoders[k].dm_idxs[f"{regr}_{val}"]]
                        except KeyError:
                            continue
                        s = bw_std[reg][:, encoders[k].dm_idxs[f"{regr}_{val}"]]

                        ax.plot(
                            encoder.tbin_centers,
                            m,
                            color=colors_strategy[k],
                            label=f"{regr}_{val}, {k}",
                        )
                        ax.fill_between(
                            encoder.tbin_centers,
                            m - s,
                            m + s,
                            color=colors_strategy[k],
                            alpha=0.25,
                        )
                        ax.axhline(y=0, linewidth=0.5, color="k")
                        ax.axvline(x=0, linewidth=0.5, color="k")

                        ax.set_ylim([-0.2, 0.6])
                        ax.legend(loc="upper right")
                        if i == len(encoder.tv_keys) + 2 - 1:
                            ax.set_xlabel("Trial Time (s)")
                        if j == 0:
                            ax.set_ylabel(r"$\beta$")
                    i += 1
    return fig, axes

In [ ]:
encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
bw_stats = {k: get_bw_stats(e) for k, e in encoders.items()}
plot_bw_traces(encoders, bw_stats)

In [ ]:
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_a = f"{regr}_{tv_vals[regr][0]}"
        regr_b = f"{regr}_{tv_vals[regr][1]}"
        assert all(
            np.isclose(
                bw_stats["full"][0]["DLS"][:, encoder.dm_idxs[regr_a]],
                bw_stats["full"][0]["DLS"][:, encoder.dm_idxs[regr_b]],
            )
        )

### aggregate across sessions

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder
from core.data import subject_ids, session_ids
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / "averaged" / subj_id
    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(f">{sess_id}")
        encoder = make_tre(Encoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
        )

        encoder_mb = make_tre(StrategyEncoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
            strategy_filter="mb",
        )

        encoder_mf = make_tre(StrategyEncoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
            strategy_filter="mf",
        )

        encoder.fit_encoder()

        try:
            encoder_mb.fit_encoder()
            encoder_mf.fit_encoder()
        except ValueError:
            continue

        encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
        bw_stats = {k: get_bw_stats(e) for k, e in encoders.items()}
        fig, _ = plot_bw_traces(encoders, bw_stats)

        save_fig(fig, fpath, fname=f"{sess_id}.png")